# Demo 06 — Decision Intelligence and Governance

This notebook converts the validated invoice into a governed business decision.

### Objectives

1. Load the validated `StructuredInvoice`.
2. Confirm eligibility for decision intelligence.
3. Discover the implemented decision interfaces.
4. Determine the business category.
5. Recommend the target finance system.
6. Calculate confidence, rationale and approval authority.
7. Apply deterministic governance policies.
8. Produce `ALLOW`, `HOLD` or `DENY`.
9. Save the decision and governance evidence.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import inspect
import json
import os
import sys

PROJECT_ROOT = Path(
    "/Users/pmayank/workspace/LdcDemo"
).resolve()

SRC_DIRECTORY = PROJECT_ROOT / "src"
SESSION_PATH = PROJECT_ROOT / "data" / "demo_session.json"
DATABASE_PATH = PROJECT_ROOT / "data" / "finance_demo.db"

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 90)
print("LOAD VALIDATED STRUCTURED INVOICE")
print("=" * 90)

assert SESSION_PATH.exists(), (
    "demo_session.json was not found. Complete Notebook 05 first."
)

demo_session = json.loads(
    SESSION_PATH.read_text(encoding="utf-8")
)

case_reference = demo_session["case_reference"]
correlation_id = demo_session["correlation_id"]
event_id = demo_session["event_id"]
invoice_filename = demo_session["invoice_filename"]

structured_extraction = demo_session.get(
    "structured_extraction",
    {},
)

structured_invoice = structured_extraction.get(
    "structured_invoice",
    {},
)

quality_review = structured_extraction.get(
    "quality_review",
    {},
)

decision_eligible = structured_extraction.get(
    "eligible_for_decision_intelligence",
    False,
)

structured_invoice_checks = {
    "Structured-extraction chapter is complete": (
        "05_STRUCTURED_EXTRACTION"
        in demo_session.get("completed_chapters", [])
    ),
    "Session status is correct": (
        demo_session.get("session_status")
        == "STRUCTURED_EXTRACTION_COMPLETED"
    ),
    "Structured invoice is available": (
        isinstance(structured_invoice, dict)
        and bool(structured_invoice)
    ),
    "Schema validation passed": (
        structured_extraction.get(
            "schema_validation_passed"
        )
        is True
    ),
    "Extraction quality gate passed": (
        quality_review.get("outcome")
        == "READY_FOR_DECISION_INTELLIGENCE"
    ),
    "Invoice is eligible for decision intelligence": (
        decision_eligible is True
    ),
}

print(f"Case reference : {case_reference}")
print(f"Correlation ID : {correlation_id}")
print(
    f"Supplier       : "
    f"{structured_invoice.get('supplier_name')}"
)
print(
    f"Invoice number : "
    f"{structured_invoice.get('invoice_number')}"
)
print(
    f"Gross amount   : "
    f"{structured_invoice.get('gross_amount')}"
)
print(
    f"Currency       : "
    f"{structured_invoice.get('currency')}"
)
print(
    f"Confidence     : "
    f"{structured_invoice.get('document_confidence')}"
)
print("-" * 90)

for check_name, passed in structured_invoice_checks.items():
    symbol = "✅" if passed else "❌"
    status = "PASS" if passed else "FAIL"
    print(f"{symbol} {status:<4} | {check_name}")

structured_invoice_ready = all(
    structured_invoice_checks.values()
)

print("-" * 90)

if structured_invoice_ready:
    print("✅ VALIDATED STRUCTURED INVOICE LOADED")
    print("The case is ready for decision intelligence.")
else:
    print("❌ CASE IS NOT ELIGIBLE FOR DECISION INTELLIGENCE")

assert structured_invoice_ready, (
    "Decision processing stopped because the structured invoice "
    "did not pass the required quality controls."
)

LOAD VALIDATED STRUCTURED INVOICE
Case reference : LOCAL-B970E86BB68E
Correlation ID : CORR-2CBC6CC15D7249CB8EE549B035F87E41
Supplier       : Northstar Marine Fuels LLC
Invoice number : NSMF-TST-260818-042
Gross amount   : 307400.0
Currency       : USD
Confidence     : 0.98
------------------------------------------------------------------------------------------
✅ PASS | Structured-extraction chapter is complete
✅ PASS | Session status is correct
✅ PASS | Structured invoice is available
✅ PASS | Schema validation passed
✅ PASS | Extraction quality gate passed
✅ PASS | Invoice is eligible for decision intelligence
------------------------------------------------------------------------------------------
✅ VALIDATED STRUCTURED INVOICE LOADED
The case is ready for decision intelligence.


In [2]:
SEARCH_TERMS = [
    "decision",
    "classification",
    "routing",
    "governance",
    "policy",
    "approval",
    "orchestration",
]

candidate_source_files = sorted({
    source_file
    for search_term in SEARCH_TERMS
    for source_file in SRC_DIRECTORY.glob(
        f"*{search_term}*.py"
    )
    if source_file.is_file()
})

print("=" * 100)
print("DECISION AND GOVERNANCE — SOURCE DISCOVERY")
print("=" * 100)

if not candidate_source_files:
    print("No matching source modules were discovered.")

for source_file in candidate_source_files:
    relative_module = (
        f"src.{source_file.stem}"
    )

    print(f"\nMODULE: {relative_module}")
    print(f"FILE  : {source_file}")
    print("-" * 100)

    source_text = source_file.read_text(
        encoding="utf-8"
    )

    syntax_tree = ast.parse(
        source_text,
        filename=str(source_file),
    )

    public_interfaces = []

    for node in syntax_tree.body:
        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        ):
            if node.name.startswith("_"):
                continue

            async_label = (
                "async "
                if isinstance(node, ast.AsyncFunctionDef)
                else ""
            )

            arguments = []

            positional_arguments = (
                node.args.posonlyargs
                + node.args.args
            )

            defaults_offset = (
                len(positional_arguments)
                - len(node.args.defaults)
            )

            for index, argument in enumerate(
                positional_arguments
            ):
                annotation = (
                    ast.unparse(argument.annotation)
                    if argument.annotation
                    else "Any"
                )

                default_index = (
                    index - defaults_offset
                )

                if default_index >= 0:
                    default_value = ast.unparse(
                        node.args.defaults[default_index]
                    )
                    arguments.append(
                        f"{argument.arg}: {annotation}="
                        f"{default_value}"
                    )
                else:
                    arguments.append(
                        f"{argument.arg}: {annotation}"
                    )

            return_type = (
                ast.unparse(node.returns)
                if node.returns
                else "Any"
            )

            public_interfaces.append(
                f"{async_label}{node.name}"
                f"({', '.join(arguments)}) "
                f"-> {return_type}"
            )

        elif isinstance(node, ast.ClassDef):
            if node.name.startswith("_"):
                continue

            public_interfaces.append(
                f"CLASS {node.name}"
            )

            for child in node.body:
                if not isinstance(
                    child,
                    (
                        ast.FunctionDef,
                        ast.AsyncFunctionDef,
                    ),
                ):
                    continue

                if child.name.startswith("_"):
                    continue

                async_label = (
                    "async "
                    if isinstance(
                        child,
                        ast.AsyncFunctionDef,
                    )
                    else ""
                )

                argument_names = [
                    argument.arg
                    for argument in (
                        child.args.posonlyargs
                        + child.args.args
                    )
                ]

                public_interfaces.append(
                    f"  {async_label}{child.name}"
                    f"({', '.join(argument_names)})"
                )

    if public_interfaces:
        for interface in public_interfaces:
            print(interface)
    else:
        print("No public functions or classes found.")

print("\n" + "=" * 100)
print(
    f"✅ DISCOVERED {len(candidate_source_files)} "
    "CANDIDATE SOURCE MODULE(S)"
)

DECISION AND GOVERNANCE — SOURCE DISCOVERY

MODULE: src.aegis_governance_adapter
FILE  : /Users/pmayank/workspace/LdcDemo/src/aegis_governance_adapter.py
----------------------------------------------------------------------------------------------------
CLASS ProposedTargetSystem
CLASS GovernanceRiskFlag
CLASS GovernanceProposal
  validate_identifier_format(cls, v)
  validate_accounting_code(cls, v)
  validate_currency(cls, v)
  validate_gross_amount(cls, v)
CLASS AegisSubmission
  non_negative(cls, v)
CLASS AegisGovernanceAdapterException
CLASS AegisGovernanceAdapter
  async submit(self, proposal)
  async check_once(self, submission)

MODULE: src.approval_workflow
FILE  : /Users/pmayank/workspace/LdcDemo/src/approval_workflow.py
----------------------------------------------------------------------------------------------------
CLASS ApprovalDecisionInput
CLASS ApprovalWorkflowResult
CLASS ApprovalWorkflow
  get_pending_approvals(self, requested_from, limit)
  get_approval_request(se

In [3]:
TARGET_SYSTEM_TERMS = [
    "VESON_IMOS",
    "SMARTPAL",
    "ORACLE_FUSION",
]

print("=" * 100)
print("TARGET-SYSTEM ROUTING INTERFACE DISCOVERY")
print("=" * 100)

routing_source_files = []

for source_file in sorted(
    SRC_DIRECTORY.rglob("*.py")
):
    source_text = source_file.read_text(
        encoding="utf-8"
    )

    if any(
        target_system in source_text
        for target_system in TARGET_SYSTEM_TERMS
    ):
        routing_source_files.append(source_file)

for source_file in routing_source_files:
    module_name = (
        "src."
        + ".".join(
            source_file.relative_to(
                SRC_DIRECTORY
            ).with_suffix("").parts
        )
    )

    print(f"\nMODULE: {module_name}")
    print(f"FILE  : {source_file}")
    print("-" * 100)

    syntax_tree = ast.parse(
        source_file.read_text(encoding="utf-8"),
        filename=str(source_file),
    )

    for node in syntax_tree.body:
        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        ):
            if node.name.startswith("_"):
                continue

            async_label = (
                "async "
                if isinstance(node, ast.AsyncFunctionDef)
                else ""
            )

            print(
                f"{async_label}"
                f"{node.name}"
                f"{ast.unparse(node.args)}"
                f" -> "
                f"{ast.unparse(node.returns) if node.returns else 'Any'}"
            )

        elif isinstance(node, ast.ClassDef):
            if node.name.startswith("_"):
                continue

            print(f"CLASS {node.name}")

            for method in node.body:
                if not isinstance(
                    method,
                    (
                        ast.FunctionDef,
                        ast.AsyncFunctionDef,
                    ),
                ):
                    continue

                if method.name.startswith("_"):
                    continue

                async_label = (
                    "async "
                    if isinstance(
                        method,
                        ast.AsyncFunctionDef,
                    )
                    else ""
                )

                print(
                    f"  {async_label}"
                    f"{method.name}"
                    f"{ast.unparse(method.args)}"
                    f" -> "
                    f"{ast.unparse(method.returns) if method.returns else 'Any'}"
                )

print("\n" + "-" * 100)
print(
    f"✅ FOUND {len(routing_source_files)} "
    "TARGET-SYSTEM SOURCE FILE(S)"
)

assert routing_source_files, (
    "No source file containing the configured target systems "
    "was discovered."
)

TARGET-SYSTEM ROUTING INTERFACE DISCOVERY

MODULE: src.aegis_governance_adapter
FILE  : /Users/pmayank/workspace/LdcDemo/src/aegis_governance_adapter.py
----------------------------------------------------------------------------------------------------
CLASS ProposedTargetSystem
CLASS GovernanceRiskFlag
CLASS GovernanceProposal
  validate_identifier_formatcls, v -> Any
  validate_accounting_codecls, v -> Any
  validate_currencycls, v -> Any
  validate_gross_amountcls, v -> Any
CLASS AegisSubmission
  non_negativecls, v -> Any
CLASS AegisGovernanceAdapterException
CLASS AegisGovernanceAdapter
  async submitself, proposal: GovernanceProposal -> AegisSubmission
  async check_onceself, submission: AegisSubmission -> AggregatedGovernanceVerdict

MODULE: src.finance_router
FILE  : /Users/pmayank/workspace/LdcDemo/src/finance_router.py
----------------------------------------------------------------------------------------------------
CLASS FinanceTargetSystem
CLASS FinanceBusinessCategory
C

## Business Classification and Finance Routing

The validated `StructuredInvoice` is submitted to the finance-routing workflow.

The router will:

1. Normalize relevant invoice information.
2. Identify business keywords.
3. Classify the finance business category.
4. Map the category to an approved target system.
5. Produce a traceable `FinanceRoutingDecision`.

Supported target systems:

- `VESON_IMOS`
- `SMARTPAL`
- `ORACLE_FUSION`

In [4]:
from dataclasses import asdict, is_dataclass
from enum import Enum
from pprint import pprint

from src.invoice_schema import StructuredInvoice
from src.finance_router import route_invoice_via_workflow


def object_to_dictionary(value):
    """Convert Pydantic, dataclass or dictionary results."""
    if isinstance(value, dict):
        return value

    if hasattr(value, "model_dump"):
        return value.model_dump()

    if hasattr(value, "to_dict"):
        return value.to_dict()

    if is_dataclass(value):
        return asdict(value)

    if hasattr(value, "__dict__"):
        return {
            key: item
            for key, item in vars(value).items()
            if not key.startswith("_")
        }

    raise TypeError(
        f"Cannot convert {type(value).__name__} to a dictionary."
    )


def enum_value(value):
    """Return the business value stored inside an Enum."""
    return value.value if isinstance(value, Enum) else value


def make_display_safe(value):
    """Recursively convert Enums and objects for readable output."""
    if isinstance(value, Enum):
        return value.value

    if isinstance(value, dict):
        return {
            key: make_display_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            make_display_safe(item)
            for item in value
        ]

    return value


print("=" * 90)
print("BUSINESS CLASSIFICATION AND FINANCE ROUTING")
print("=" * 90)

validated_invoice_model = StructuredInvoice(
    **structured_invoice
)

print(f"Case reference : {case_reference}")
print(
    f"Supplier       : "
    f"{validated_invoice_model.supplier_name}"
)
print(
    f"Invoice number : "
    f"{validated_invoice_model.invoice_number}"
)
print(
    f"Gross amount   : "
    f"{validated_invoice_model.gross_amount}"
)
print(
    f"Currency       : "
    f"{validated_invoice_model.currency}"
)
print("-" * 90)
print("Executing deterministic finance-routing workflow...")

routing_decision = route_invoice_via_workflow(
    validated_invoice_model
)

routing_decision_dict = object_to_dictionary(
    routing_decision
)

routing_display = make_display_safe(
    routing_decision_dict
)

print("✅ FINANCE ROUTING WORKFLOW COMPLETED")
print("\nRouting decision:")
pprint(routing_display, sort_dicts=False)

BUSINESS CLASSIFICATION AND FINANCE ROUTING
Case reference : LOCAL-B970E86BB68E
Supplier       : Northstar Marine Fuels LLC
Invoice number : NSMF-TST-260818-042
Gross amount   : 307400.0
Currency       : USD
------------------------------------------------------------------------------------------
Executing deterministic finance-routing workflow...


/Users/pmayank/workspace/LdcDemo/src/finance_router.py:334: UserWarning: Instance-based API usage detected. Consider using preferred pattern:
  CURRENT: add_node(<instance>, 'classify_finance_route')
  PREFERRED: add_node('HandlerNode', 'classify_finance_route', {'param': value})
  node_id = builder.add_node(handler_node, "classify_finance_route")
[NODE] Unknown parameter(s) for HandlerNode: ['handler', 'params']. Valid parameters: [].
[NODE] Unknown parameter(s) for HandlerNode: ['handler', 'params']. Valid parameters: [].


✅ FINANCE ROUTING WORKFLOW COMPLETED

Routing decision:
{'target_system': 'VESON_IMOS',
 'routing_status': 'ROUTE_CANDIDATE',
 'reason_code': 'SINGLE_CATEGORY_MATCH',
 'matched_categories': ['BUNKER'],
 'matched_evidence_terms': ['bunker'],
 'route_confidence': 0.9000000000000001,
 'explanation': "Categories ['BUNKER'] map to VESON_IMOS",
 'workflow_run_id': 'bacf24ad-1d74-4142-be0e-d489b698a32c'}


In [8]:
SUPPORTED_TARGET_SYSTEMS = {
    "VESON_IMOS",
    "SMARTPAL",
    "ORACLE_FUSION",
}


def first_available(mapping, field_names):
    for field_name in field_names:
        if field_name in mapping:
            return enum_value(mapping[field_name])

    return None


routing_status = enum_value(
    routing_decision_dict["routing_status"]
)

target_system = enum_value(
    routing_decision_dict["target_system"]
)

routing_reason_code = enum_value(
    routing_decision_dict["reason_code"]
)

matched_categories = [
    enum_value(category)
    for category in routing_decision_dict[
        "matched_categories"
    ]
]

matched_evidence_terms = list(
    routing_decision_dict[
        "matched_evidence_terms"
    ]
)

routing_confidence = float(
    routing_decision_dict["route_confidence"]
)

routing_explanation = routing_decision_dict[
    "explanation"
]

routing_workflow_run_id = routing_decision_dict[
    "workflow_run_id"
]

routing_checks = {
    "Routing status is ROUTE_CANDIDATE": (
        routing_status == "ROUTE_CANDIDATE"
    ),
    "Business category is BUNKER": (
        "BUNKER" in matched_categories
    ),
    "Target system is VESON_IMOS": (
        target_system == "VESON_IMOS"
    ),
    "Reason is SINGLE_CATEGORY_MATCH": (
        routing_reason_code
        == "SINGLE_CATEGORY_MATCH"
    ),
    "Routing evidence is available": (
        bool(matched_evidence_terms)
    ),
    "Routing confidence is valid": (
        0 <= routing_confidence <= 1
    ),
    "Routing explanation is available": (
        bool(routing_explanation)
    ),
    "Routing workflow ID is available": (
        bool(routing_workflow_run_id)
    ),
}

print("=" * 90)
print("ROUTING DECISION SUMMARY")
print("=" * 90)
print(f"Routing status    : {routing_status}")
print(f"Business category : {matched_categories}")
print(f"Target system     : {target_system}")
print(f"Reason code       : {routing_reason_code}")
print(
    f"Confidence        : "
    f"{routing_confidence:.2%}"
)
print(
    f"Evidence terms    : "
    f"{matched_evidence_terms}"
)
print(f"Explanation       : {routing_explanation}")
print(
    f"Workflow run ID   : "
    f"{routing_workflow_run_id}"
)
print("-" * 90)

for check_name, passed in routing_checks.items():
    symbol = "✅" if passed else "❌"
    status = "PASS" if passed else "FAIL"
    print(f"{symbol} {status:<4} | {check_name}")

routing_decision_verified = all(
    routing_checks.values()
)

print("-" * 90)

if routing_decision_verified:
    print("✅ BUSINESS CATEGORY: BUNKER")
    print("✅ TARGET-SYSTEM RECOMMENDATION: VESON_IMOS")
    print("✅ ROUTING DECISION VERIFIED")
else:
    print("❌ ROUTING DECISION VALIDATION FAILED")

assert routing_decision_verified, (
    "The finance-routing decision did not pass validation."
)

ROUTING DECISION SUMMARY
Routing status    : ROUTE_CANDIDATE
Business category : ['BUNKER']
Target system     : VESON_IMOS
Reason code       : SINGLE_CATEGORY_MATCH
Confidence        : 90.00%
Evidence terms    : ['bunker']
Explanation       : Categories ['BUNKER'] map to VESON_IMOS
Workflow run ID   : bacf24ad-1d74-4142-be0e-d489b698a32c
------------------------------------------------------------------------------------------
✅ PASS | Routing status is ROUTE_CANDIDATE
✅ PASS | Business category is BUNKER
✅ PASS | Target system is VESON_IMOS
✅ PASS | Reason is SINGLE_CATEGORY_MATCH
✅ PASS | Routing evidence is available
✅ PASS | Routing confidence is valid
✅ PASS | Routing explanation is available
✅ PASS | Routing workflow ID is available
------------------------------------------------------------------------------------------
✅ BUSINESS CATEGORY: BUNKER
✅ TARGET-SYSTEM RECOMMENDATION: VESON_IMOS
✅ ROUTING DECISION VERIFIED


> ### The validated invoice now enters deterministic finance routing. The workflow identifies the business category from the extracted invoice facts and maps it to an approved finance platform. Because this is a maritime bunker-related invoice, we expect the recommended destination to be VESON IMOS. The routing output remains a proposal—it must still pass governance before any posting can occur.”

## Approval-Authority Decision

Finance routing determines where the invoice should go.

Approval intelligence determines:

- Whether approval is required
- Required authority level
- Approval reason
- Whether an approval has already been provided

Governance evaluates both the routing proposal and approval decision before returning `ALLOW`, `HOLD` or `DENY`.

In [10]:
print("=" * 100)
print("APPROVAL-DECISION CONTRACT DISCOVERY")
print("=" * 100)

approval_source_files = []

for source_file in sorted(
    SRC_DIRECTORY.rglob("*.py")
):
    source_text = source_file.read_text(
        encoding="utf-8"
    )

    if "ApprovalDecision" in source_text:
        approval_source_files.append(source_file)

for source_file in approval_source_files:
    module_name = (
        "src."
        + ".".join(
            source_file.relative_to(
                SRC_DIRECTORY
            ).with_suffix("").parts
        )
    )

    print(f"\nMODULE: {module_name}")
    print(f"FILE  : {source_file}")
    print("-" * 100)

    syntax_tree = ast.parse(
        source_file.read_text(encoding="utf-8"),
        filename=str(source_file),
    )

    for node in syntax_tree.body:
        if isinstance(node, ast.ClassDef):
            if (
                "Approval" not in node.name
                and "Authority" not in node.name
            ):
                continue

            print(f"CLASS {node.name}")

            annotated_fields = [
                child
                for child in node.body
                if isinstance(child, ast.AnnAssign)
            ]

            for field in annotated_fields:
                field_name = (
                    field.target.id
                    if isinstance(field.target, ast.Name)
                    else ast.unparse(field.target)
                )

                field_type = ast.unparse(
                    field.annotation
                )

                default_value = (
                    ast.unparse(field.value)
                    if field.value is not None
                    else "REQUIRED"
                )

                print(
                    f"  FIELD {field_name}: "
                    f"{field_type} = {default_value}"
                )

            for method in node.body:
                if not isinstance(
                    method,
                    (
                        ast.FunctionDef,
                        ast.AsyncFunctionDef,
                    ),
                ):
                    continue

                if method.name.startswith("_"):
                    continue

                async_label = (
                    "async "
                    if isinstance(
                        method,
                        ast.AsyncFunctionDef,
                    )
                    else ""
                )

                print(
                    f"  {async_label}"
                    f"{method.name}"
                    f"{ast.unparse(method.args)}"
                    f" -> "
                    f"{ast.unparse(method.returns) if method.returns else 'Any'}"
                )

        elif isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        ):
            source_segment = ast.get_source_segment(
                source_file.read_text(encoding="utf-8"),
                node,
            ) or ""

            if (
                "ApprovalDecision" not in source_segment
                and "approval" not in node.name.lower()
                and "authority" not in node.name.lower()
            ):
                continue

            async_label = (
                "async "
                if isinstance(node, ast.AsyncFunctionDef)
                else ""
            )

            print(
                f"{async_label}"
                f"{node.name}"
                f"{ast.unparse(node.args)}"
                f" -> "
                f"{ast.unparse(node.returns) if node.returns else 'Any'}"
            )

print("\n" + "-" * 100)
print(
    f"✅ FOUND {len(approval_source_files)} "
    "APPROVAL-RELATED MODULE(S)"
)

assert approval_source_files, (
    "No module containing ApprovalDecision was found."
)

APPROVAL-DECISION CONTRACT DISCOVERY

MODULE: src.approval_workflow
FILE  : /Users/pmayank/workspace/LdcDemo/src/approval_workflow.py
----------------------------------------------------------------------------------------------------
CLASS ApprovalDecisionInput
  FIELD approval_request_id: str = Field(..., min_length=1, max_length=64)
  FIELD approver_id: str = Field(..., min_length=1, max_length=64)
  FIELD approver_authority: str = Field(..., min_length=1, max_length=64)
  FIELD decision: str = Field(..., pattern='^(APPROVED|REJECTED)$')
  FIELD approver_comment: str = ''
CLASS ApprovalWorkflowResult
  FIELD approval_request_id: str = Field(..., min_length=1, max_length=64)
  FIELD invoice_id: str = Field(..., min_length=1, max_length=64)
  FIELD correlation_id: str = Field(..., min_length=1, max_length=64)
  FIELD decision: str = REQUIRED
  FIELD approver_id: str = REQUIRED
  FIELD invoice_case_status: str = REQUIRED
  FIELD approval_request_status: str = REQUIRED
  FIELD decided_a

## Approval-Authority Calculation

The authority hierarchy calculates who must approve the invoice based on:

- Gross amount
- Currency
- Identified risk flags
- Multi-approval requirements

The result includes:

- Required authority levels
- Approval reasons
- Highest risk level
- Risk count
- Multi-approval requirement
- Deterministic decision hash

This step calculates the required approval route; it does not record a human approval.

In [11]:
from src.authority_hierarchy import (
    FinanceAuthorityHierarchy,
)

gross_amount = structured_invoice.get(
    "gross_amount"
)

currency = structured_invoice.get(
    "currency",
    "USD",
)

assert gross_amount is not None, (
    "Gross amount is required for approval-authority calculation."
)

assert currency, (
    "Currency is required for approval-authority calculation."
)

# Build explicit risk signals from validated invoice facts.
approval_risk_flags = []

if structured_invoice.get("bank_change_claimed") is True:
    approval_risk_flags.append(
        "BANK_CHANGE_CLAIMED"
    )

if structured_invoice.get("requires_human_review") is True:
    approval_risk_flags.append(
        "EXTRACTION_HUMAN_REVIEW_REQUIRED"
    )

requires_multi_approval_input = False

print("=" * 90)
print("APPROVAL-AUTHORITY CALCULATION")
print("=" * 90)
print(f"Invoice ID              : {case_reference}")
print(f"Correlation ID          : {correlation_id}")
print(f"Gross amount            : {gross_amount}")
print(f"Currency                : {currency}")
print(
    f"Risk flags              : "
    f"{approval_risk_flags or 'None'}"
)
print(
    f"Multi-approval requested: "
    f"{requires_multi_approval_input}"
)
print("-" * 90)

authority_hierarchy = FinanceAuthorityHierarchy()

approval_decision = (
    authority_hierarchy.determine_approval_decision(
        invoice_id=case_reference,
        correlation_id=correlation_id,
        gross_amount=float(gross_amount),
        currency=currency,
        risk_flags=approval_risk_flags,
        requires_multi_approval=(
            requires_multi_approval_input
        ),
    )
)

approval_decision_dict = object_to_dictionary(
    approval_decision
)

approval_display = make_display_safe(
    approval_decision_dict
)

print("Approval decision:")
pprint(approval_display, sort_dicts=False)

APPROVAL-AUTHORITY CALCULATION
Invoice ID              : LOCAL-B970E86BB68E
Correlation ID          : CORR-2CBC6CC15D7249CB8EE549B035F87E41
Gross amount            : 307400.0
Currency                : USD
Risk flags              : None
Multi-approval requested: False
------------------------------------------------------------------------------------------
Approval decision:
{'invoice_id': 'LOCAL-B970E86BB68E',
 'correlation_id': 'CORR-2CBC6CC15D7249CB8EE549B035F87E41',
 'required_authorities': ['L3_CONTROLLER'],
 'approval_reasons': ['AMOUNT_THRESHOLD'],
 'highest_risk_level': 'MEDIUM',
 'total_risk_count': 1,
 'requires_multi_approval': False,
 'approval_chain_notes': 'Amount $307400.00 exceeds clerk threshold',
 'deterministic_hash': 'dd7e01084edc7dfb'}


In [12]:
required_authorities = [
    enum_value(authority)
    for authority in approval_decision_dict.get(
        "required_authorities",
        [],
    )
]

approval_reasons = [
    enum_value(reason)
    for reason in approval_decision_dict.get(
        "approval_reasons",
        [],
    )
]

highest_risk_level = enum_value(
    approval_decision_dict.get(
        "highest_risk_level"
    )
)

total_risk_count = approval_decision_dict.get(
    "total_risk_count",
    0,
)

requires_multi_approval = bool(
    approval_decision_dict.get(
        "requires_multi_approval",
        False,
    )
)

approval_chain_notes = approval_decision_dict.get(
    "approval_chain_notes",
    "",
)

approval_deterministic_hash = (
    approval_decision_dict.get(
        "deterministic_hash",
        "",
    )
)

approval_checks = {
    "ApprovalDecision was returned": (
        approval_decision is not None
    ),
    "Invoice ID is preserved": (
        approval_decision_dict.get("invoice_id")
        == case_reference
    ),
    "Correlation ID is preserved": (
        approval_decision_dict.get("correlation_id")
        == correlation_id
    ),
    "Risk level was calculated": (
        bool(highest_risk_level)
    ),
    "Risk count is valid": (
        isinstance(total_risk_count, int)
        and total_risk_count >= 0
    ),
    "Deterministic hash is available": (
        isinstance(approval_deterministic_hash, str)
        and bool(approval_deterministic_hash)
    ),
}

print("\n" + "=" * 90)
print("APPROVAL DECISION SUMMARY")
print("=" * 90)
print(
    f"Required authorities : "
    f"{required_authorities or 'None'}"
)
print(
    f"Approval reasons     : "
    f"{approval_reasons or 'None'}"
)
print(f"Highest risk level   : {highest_risk_level}")
print(f"Total risk count     : {total_risk_count}")
print(
    f"Multi-approval       : "
    f"{requires_multi_approval}"
)
print(f"Approval-chain notes : {approval_chain_notes}")
print(
    f"Deterministic hash   : "
    f"{approval_deterministic_hash}"
)
print("-" * 90)

for check_name, passed in approval_checks.items():
    symbol = "✅" if passed else "❌"
    status = "PASS" if passed else "FAIL"
    print(f"{symbol} {status:<4} | {check_name}")

approval_decision_verified = all(
    approval_checks.values()
)

print("-" * 90)

if approval_decision_verified:
    print("✅ APPROVAL AUTHORITY CALCULATED")
    print("✅ DETERMINISTIC APPROVAL DECISION VERIFIED")
else:
    print("❌ APPROVAL-AUTHORITY VALIDATION FAILED")

assert approval_decision_verified, (
    "The approval-authority decision did not pass validation."
)


APPROVAL DECISION SUMMARY
Required authorities : ['L3_CONTROLLER']
Approval reasons     : ['AMOUNT_THRESHOLD']
Highest risk level   : MEDIUM
Total risk count     : 1
Multi-approval       : False
Approval-chain notes : Amount $307400.00 exceeds clerk threshold
Deterministic hash   : dd7e01084edc7dfb
------------------------------------------------------------------------------------------
✅ PASS | ApprovalDecision was returned
✅ PASS | Invoice ID is preserved
✅ PASS | Correlation ID is preserved
✅ PASS | Risk level was calculated
✅ PASS | Risk count is valid
✅ PASS | Deterministic hash is available
------------------------------------------------------------------------------------------
✅ APPROVAL AUTHORITY CALCULATED
✅ DETERMINISTIC APPROVAL DECISION VERIFIED


The routing agent decides where the invoice belongs, while the authority hierarchy decides who must approve it. This calculation is deterministic and based on the financial amount, currency and explicit risk signals. The resulting decision includes a tamper-evident hash, giving us a repeatable and auditable approval requirement.

#### Before executing governance, we need one small interface check because GovernanceOrchestrator may require an Aegis adapter or other dependencies in its constructor.

In [13]:
from src.governance_orchestrator import (
    GovernanceOrchestrator,
)
from src.aegis_governance_adapter import (
    AegisGovernanceAdapter,
)

print("=" * 90)
print("GOVERNANCE ORCHESTRATION INTERFACE")
print("=" * 90)

print(
    "GovernanceOrchestrator constructor:"
)
print(
    inspect.signature(GovernanceOrchestrator)
)

print("\nOrchestration method:")
print(
    inspect.signature(
        GovernanceOrchestrator.orchestrate
    )
)

print("\nAegisGovernanceAdapter constructor:")
print(
    inspect.signature(AegisGovernanceAdapter)
)

print("\nAdapter methods:")
print(
    "submit:",
    inspect.signature(
        AegisGovernanceAdapter.submit
    ),
)
print(
    "check_once:",
    inspect.signature(
        AegisGovernanceAdapter.check_once
    ),
)

print("-" * 90)
print("✅ GOVERNANCE INTERFACE INSPECTED")

GOVERNANCE ORCHESTRATION INTERFACE
GovernanceOrchestrator constructor:
(adapter: src.aegis_governance_adapter.AegisGovernanceAdapter)

Orchestration method:
(self, routing_decision: src.finance_router.FinanceRoutingDecision, approval_decision: src.authority_hierarchy.ApprovalDecision, correlation_id: str, case_id: str) -> src.governance_orchestrator.GovernanceOrchestrationResult

AegisGovernanceAdapter constructor:
(client)

Adapter methods:
submit: (self, proposal: src.aegis_governance_adapter.GovernanceProposal) -> src.aegis_governance_adapter.AegisSubmission
check_once: (self, submission: src.aegis_governance_adapter.AegisSubmission) -> src.governance_contract.AggregatedGovernanceVerdict
------------------------------------------------------------------------------------------
✅ GOVERNANCE INTERFACE INSPECTED


## Discover Existing Aegis Client Wiring

In [14]:
from src.invoice_orchestration_service import (
    InvoiceOrchestrationService,
)

print("=" * 100)
print("AEGIS CLIENT-WIRING DISCOVERY")
print("=" * 100)

print(
    "InvoiceOrchestrationService constructor:"
)
print(
    inspect.signature(InvoiceOrchestrationService)
)

print("\nConstructor source:")
print("-" * 100)

try:
    print(
        inspect.getsource(
            InvoiceOrchestrationService.__init__
        )
    )
except (OSError, TypeError):
    print("Constructor source unavailable.")

print("\n" + "=" * 100)
print("ADAPTER/ORCHESTRATOR CONSTRUCTION LOCATIONS")
print("=" * 100)

search_patterns = [
    "AegisGovernanceAdapter(",
    "GovernanceOrchestrator(",
]

matches_found = 0

for source_file in sorted(
    SRC_DIRECTORY.rglob("*.py")
):
    source_lines = source_file.read_text(
        encoding="utf-8"
    ).splitlines()

    for line_number, line in enumerate(
        source_lines,
        start=1,
    ):
        if not any(
            pattern in line
            for pattern in search_patterns
        ):
            continue

        matches_found += 1

        print(
            f"\n{source_file.name}:{line_number}"
        )
        print("-" * 100)

        start_line = max(1, line_number - 6)
        end_line = min(
            len(source_lines),
            line_number + 8,
        )

        for context_line_number in range(
            start_line,
            end_line + 1,
        ):
            marker = (
                ">>"
                if context_line_number == line_number
                else "  "
            )

            print(
                f"{marker} "
                f"{context_line_number:04d}: "
                f"{source_lines[context_line_number - 1]}"
            )

print("\n" + "-" * 100)
print(f"Construction locations found: {matches_found}")

AEGIS CLIENT-WIRING DISCOVERY
InvoiceOrchestrationService constructor:
(aegis_adapter: Optional[src.aegis_governance_adapter.AegisGovernanceAdapter] = None)

Constructor source:
----------------------------------------------------------------------------------------------------
    def __init__(self, aegis_adapter: Optional[AegisGovernanceAdapter] = None):
        """Initialize orchestration service with required dependencies.

        Args:
            aegis_adapter: Injected AegisGovernanceAdapter (mocked for tests)
        """
        self.kill_switch = KillSwitchService()
        self.audit_chain = AuditChainService()
        self.aegis_adapter = aegis_adapter
        self.governance_orchestrator = (
            GovernanceOrchestrator(aegis_adapter) if aegis_adapter else None
        )


ADAPTER/ORCHESTRATOR CONSTRUCTION LOCATIONS

invoice_orchestration_service.py:79
----------------------------------------------------------------------------------------------------
   0073:       

In [15]:
from src.aegis_governance_adapter import (
    AegisSubmission,
)
from src.governance_contract import (
    AggregatedGovernanceVerdict,
    RequestGovernanceResult,
)
from src.governance_orchestrator import (
    GovernanceOrchestrationResult,
)

print("=" * 100)
print("GOVERNANCE MODEL CONTRACTS")
print("=" * 100)

models_to_inspect = [
    AegisSubmission,
    RequestGovernanceResult,
    AggregatedGovernanceVerdict,
    GovernanceOrchestrationResult,
]

for model_class in models_to_inspect:
    print(f"\n{model_class.__name__}")
    print("-" * 100)
    print(f"Constructor: {inspect.signature(model_class)}")

    model_fields = getattr(
        model_class,
        "model_fields",
        None,
    )

    dataclass_fields = getattr(
        model_class,
        "__dataclass_fields__",
        None,
    )

    if model_fields:
        for field_name, field_info in model_fields.items():
            print(
                f"  {field_name:<32} "
                f"type={field_info.annotation}"
            )

    elif dataclass_fields:
        for field_name, field_info in dataclass_fields.items():
            print(
                f"  {field_name:<32} "
                f"type={field_info.type}"
            )

print("\n" + "=" * 100)
print("GOVERNANCE ORCHESTRATION LOGIC")
print("=" * 100)

print(
    inspect.getsource(
        GovernanceOrchestrator.orchestrate
    )
)

GOVERNANCE MODEL CONTRACTS

AegisSubmission
----------------------------------------------------------------------------------------------------
Constructor: (*, objective_id: Annotated[str, MinLen(min_length=1)], objective_status: str = '', execution_status: str = '', expected_request_count: int = 0, correlation_id: Annotated[str, MinLen(min_length=1)]) -> None
  objective_id                     type=<class 'str'>
  objective_status                 type=<class 'str'>
  execution_status                 type=<class 'str'>
  expected_request_count           type=<class 'int'>
  correlation_id                   type=<class 'str'>

RequestGovernanceResult
----------------------------------------------------------------------------------------------------
Constructor: (*, request_id: Annotated[str, MinLen(min_length=1)], raw_decision_type: Optional[str] = None, action: src.governance_contract.GovernanceAction, reason_code: src.governance_contract.GovernanceReason, reason_detail: str = '', c

> ### The validated invoice now enters deterministic finance routing. The workflow identifies the business category from the extracted invoice facts and maps it to an approved finance platform. Because this is a maritime bunker-related invoice, we expect the recommended destination to be VESON IMOS. The routing output remains a proposal—it must still pass governance before any posting can occur.”

In [16]:
from src.governance_contract import (
    AggregatedGovernanceVerdict,
)

print("=" * 90)
print("FINAL GOVERNANCE INPUTS")
print("=" * 90)

print("\nAPPROVAL DECISION")
print(f"Required authorities : {required_authorities}")
print(f"Approval reasons     : {approval_reasons}")
print(f"Highest risk level   : {highest_risk_level}")
print(f"Total risk count     : {total_risk_count}")
print(f"Multi-approval       : {requires_multi_approval}")
print(f"Approval notes       : {approval_chain_notes}")

print("\nAGGREGATED VERDICT CONTRACT")
print(
    inspect.signature(
        AggregatedGovernanceVerdict
    )
)

model_fields = getattr(
    AggregatedGovernanceVerdict,
    "model_fields",
    {},
)

for field_name, field_info in model_fields.items():
    print(
        f"{field_name:<30} "
        f"type={field_info.annotation} "
        f"required={field_info.is_required()}"
    )

FINAL GOVERNANCE INPUTS

APPROVAL DECISION
Required authorities : ['L3_CONTROLLER']
Approval reasons     : ['AMOUNT_THRESHOLD']
Highest risk level   : MEDIUM
Total risk count     : 1
Multi-approval       : False
Approval notes       : Amount $307400.00 exceeds clerk threshold

AGGREGATED VERDICT CONTRACT
(*, action: src.governance_contract.GovernanceAction, reason_code: src.governance_contract.GovernanceReason, reason_detail: str = '', objective_id: Annotated[str, MinLen(min_length=1)], request_results: List[src.governance_contract.RequestGovernanceResult] = <factory>, all_requests_final: bool = False, expected_request_count: int = 0, observed_request_count: int = 0) -> None
action                         type=<enum 'GovernanceAction'> required=True
reason_code                    type=<enum 'GovernanceReason'> required=True
reason_detail                  type=<class 'str'> required=False
objective_id                   type=<class 'str'> required=True
request_results                type

## Governance Orchestration

The real governance orchestrator evaluates:

1. The proposed `VESON_IMOS` route
2. Routing confidence
3. Invoice risk level
4. Required approval authority
5. Aegis policy response

For this local demonstration, the external Aegis response is controlled and repeatable:

`conditional_approval` → `HOLD`

This allows the next chapter to demonstrate human approval before posting.

In [18]:
from src.aegis_governance_adapter import (
    AegisSubmission,
)
from src.governance_contract import (
    RequestGovernanceResult,
    aggregate_governance_verdicts,
    map_raw_decision_to_action,
    map_raw_decision_to_reason,
)
from src.governance_orchestrator import (
    GovernanceOrchestrator,
)


class ControlledDemoAegisAdapter:
    """
    Predictable local adapter used to demonstrate governance
    without depending on a live external Aegis connection.
    """

    def __init__(self):
        self.last_proposal = None
        self.last_submission = None

    async def submit(self, proposal):
        self.last_proposal = proposal

        submission = AegisSubmission(
            objective_id=(
                f"OBJ-{case_reference}"
            ),
            objective_status="submitted",
            execution_status="completed",
            expected_request_count=1,
            correlation_id=correlation_id,
        )

        self.last_submission = submission
        return submission

    async def check_once(self, submission):
        raw_decision = "conditional_approval"

        request_result = RequestGovernanceResult(
            request_id=(
                f"REQ-{case_reference}"
            ),
            raw_decision_type=raw_decision,
            action=map_raw_decision_to_action(
                raw_decision
            ),
            reason_code=map_raw_decision_to_reason(
                raw_decision
            ),
            reason_detail=(
                "Invoice requires L3 Controller approval "
                "because the amount exceeds the clerk threshold."
            ),
            conditions_json=json.dumps(
                {
                    "required_authority": (
                        "L3_CONTROLLER"
                    ),
                    "reason": "AMOUNT_THRESHOLD",
                    "gross_amount": float(gross_amount),
                    "currency": currency,
                },
                sort_keys=True,
            ),
            review_decision_id=(
                f"REV-{case_reference}"
            ),
        )

        return aggregate_governance_verdicts(
            request_results=[request_result],
            objective_id=submission.objective_id,
            expected_request_count=1,
        )


print("=" * 90)
print("GOVERNANCE ORCHESTRATION")
print("=" * 90)
print(f"Case reference    : {case_reference}")
print(f"Correlation ID    : {correlation_id}")
print(f"Proposed system   : {target_system}")
print(f"Business category : {matched_categories}")
print(f"Routing confidence: {routing_confidence:.2%}")
print(f"Required authority: {required_authorities}")
print(f"Risk level        : {highest_risk_level}")
print("-" * 90)

controlled_aegis_adapter = (
    ControlledDemoAegisAdapter()
)

governance_orchestrator = GovernanceOrchestrator(
    controlled_aegis_adapter
)

governance_result = await (
    governance_orchestrator.orchestrate(
        routing_decision=routing_decision,
        approval_decision=approval_decision,
        correlation_id=correlation_id,
        case_id=case_reference,
    )
)

governance_result_dict = object_to_dictionary(
    governance_result
)

governance_display = make_display_safe(
    governance_result_dict
)

print("Governance result:")
pprint(governance_display, sort_dicts=False)

GOVERNANCE ORCHESTRATION
Case reference    : LOCAL-B970E86BB68E
Correlation ID    : CORR-2CBC6CC15D7249CB8EE549B035F87E41
Proposed system   : VESON_IMOS
Business category : ['BUNKER']
Routing confidence: 90.00%
Required authority: ['L3_CONTROLLER']
Risk level        : MEDIUM
------------------------------------------------------------------------------------------
Governance result:
{'case_id': 'LOCAL-B970E86BB68E',
 'correlation_id': 'CORR-2CBC6CC15D7249CB8EE549B035F87E41',
 'proposed_target_system': 'VESON_IMOS',
 'evaluating_authority': 'L3_CONTROLLER',
 'required_authority': 'L3_CONTROLLER',
 'authority_outcome': 'sufficient',
 'governance_status': 'HOLD',
 'reason_code': 'GOVERNANCE_PENDING',
 'objective_id': 'OBJ-LOCAL-B970E86BB68E',
 'request_ids': ['REQ-LOCAL-B970E86BB68E'],
 'review_decision_ids': ['REV-LOCAL-B970E86BB68E'],
 'routing_workflow_run_id': 'bacf24ad-1d74-4142-be0e-d489b698a32c',
 'explanation': 'Aegis holding: Request REQ-LOCAL-B970E86BB68E has conditional '
     

In [19]:
governance_status = enum_value(
    governance_result_dict.get(
        "governance_status"
    )
)

governance_reason = enum_value(
    governance_result_dict.get(
        "reason_code"
    )
)

governance_target_system = (
    governance_result_dict.get(
        "proposed_target_system"
    )
)

governance_required_authority = (
    governance_result_dict.get(
        "required_authority"
    )
)

governance_authority_outcome = (
    governance_result_dict.get(
        "authority_outcome"
    )
)

governance_objective_id = (
    governance_result_dict.get(
        "objective_id"
    )
)

governance_request_ids = (
    governance_result_dict.get(
        "request_ids",
        [],
    )
)

governance_review_decision_ids = (
    governance_result_dict.get(
        "review_decision_ids",
        [],
    )
)

governance_explanation = (
    governance_result_dict.get(
        "explanation",
        "",
    )
)

governance_checks = {
    "Governance result was returned": (
        governance_result is not None
    ),
    "Case reference is preserved": (
        governance_result_dict.get("case_id")
        == case_reference
    ),
    "Correlation ID is preserved": (
        governance_result_dict.get(
            "correlation_id"
        )
        == correlation_id
    ),
    "Target system is VESON_IMOS": (
        governance_target_system
        == "VESON_IMOS"
    ),
    "Governance outcome is HOLD": (
        governance_status == "HOLD"
    ),
    "Governance reason is available": (
        bool(governance_reason)
    ),
    "Aegis objective ID is available": (
        bool(governance_objective_id)
    ),
    "Governance explanation is available": (
        bool(governance_explanation)
    ),
}

print("\n" + "=" * 90)
print("GOVERNANCE DECISION SUMMARY")
print("=" * 90)
print(f"Governance status : {governance_status}")
print(f"Reason code       : {governance_reason}")
print(
    f"Target system     : "
    f"{governance_target_system}"
)
print(
    f"Required authority: "
    f"{governance_required_authority}"
)
print(
    f"Authority outcome : "
    f"{governance_authority_outcome}"
)
print(
    f"Objective ID      : "
    f"{governance_objective_id}"
)
print(
    f"Request IDs       : "
    f"{governance_request_ids}"
)
print(
    f"Review IDs        : "
    f"{governance_review_decision_ids}"
)
print(f"Explanation       : {governance_explanation}")
print("-" * 90)

for check_name, passed in governance_checks.items():
    symbol = "✅" if passed else "❌"
    status = "PASS" if passed else "FAIL"
    print(f"{symbol} {status:<4} | {check_name}")

governance_decision_verified = all(
    governance_checks.values()
)

print("-" * 90)

if governance_decision_verified:
    print("✅ GOVERNANCE OUTCOME: HOLD")
    print("✅ HUMAN APPROVAL REQUIRED BEFORE POSTING")
else:
    print("❌ GOVERNANCE VALIDATION FAILED")

assert governance_decision_verified, (
    "The governance orchestration did not produce "
    "the expected controlled HOLD."
)


GOVERNANCE DECISION SUMMARY
Governance status : HOLD
Reason code       : GOVERNANCE_PENDING
Target system     : VESON_IMOS
Required authority: L3_CONTROLLER
Authority outcome : sufficient
Objective ID      : OBJ-LOCAL-B970E86BB68E
Request IDs       : ['REQ-LOCAL-B970E86BB68E']
Review IDs        : ['REV-LOCAL-B970E86BB68E']
Explanation       : Aegis holding: Request REQ-LOCAL-B970E86BB68E has conditional approval
------------------------------------------------------------------------------------------
✅ PASS | Governance result was returned
✅ PASS | Case reference is preserved
✅ PASS | Correlation ID is preserved
✅ PASS | Target system is VESON_IMOS
✅ PASS | Governance outcome is HOLD
✅ PASS | Governance reason is available
✅ PASS | Aegis objective ID is available
✅ PASS | Governance explanation is available
------------------------------------------------------------------------------------------
✅ GOVERNANCE OUTCOME: HOLD
✅ HUMAN APPROVAL REQUIRED BEFORE POSTING


“The invoice has a valid route to VESON IMOS, but its value of approximately $307,400 exceeds the clerk threshold. Aegis therefore returns conditional approval, and the governance orchestrator places the case on HOLD for an L3 Controller. The system has made progress autonomously, but it cannot post the invoice until the correct human authority approves it.

In [20]:
from src.governance_persistence import (
    GovernancePersistenceService,
    PersistenceOutcome,
    ApprovalRequestCreated,
)

print("=" * 100)
print("GOVERNANCE PERSISTENCE CONTRACT")
print("=" * 100)

print(
    "GovernancePersistenceService constructor:"
)
print(
    inspect.signature(
        GovernancePersistenceService
    )
)

print("\nPersistence methods:")
print(
    "persist_decision:",
    inspect.signature(
        GovernancePersistenceService.persist_decision
    ),
)
print(
    "persist_hold_decision:",
    inspect.signature(
        GovernancePersistenceService.persist_hold_decision
    ),
)

for model_class in [
    PersistenceOutcome,
    ApprovalRequestCreated,
]:
    print(f"\nMODEL: {model_class.__name__}")
    print(
        f"Constructor: "
        f"{inspect.signature(model_class)}"
    )

    model_fields = getattr(
        model_class,
        "model_fields",
        None,
    )

    dataclass_fields = getattr(
        model_class,
        "__dataclass_fields__",
        None,
    )

    if model_fields:
        for field_name, field_info in model_fields.items():
            print(
                f"  {field_name:<32} "
                f"type={field_info.annotation}"
            )

    elif dataclass_fields:
        for field_name, field_info in dataclass_fields.items():
            print(
                f"  {field_name:<32} "
                f"type={field_info.type}"
            )

print("-" * 100)
print("✅ GOVERNANCE PERSISTENCE INTERFACE INSPECTED")

GOVERNANCE PERSISTENCE CONTRACT
GovernancePersistenceService constructor:
()

Persistence methods:
persist_decision: (self, orchestration_result: src.governance_orchestrator.GovernanceOrchestrationResult, case_id: str, correlation_id: str, invoice_case_id: Optional[str] = None) -> src.governance_persistence.PersistenceOutcome
persist_hold_decision: (self, orchestration_result: src.governance_orchestrator.GovernanceOrchestrationResult, case_id: str, correlation_id: str, finance_decision_id: Optional[str] = None) -> Optional[src.governance_persistence.ApprovalRequestCreated]

MODEL: PersistenceOutcome
Constructor: (*, outcome_status: Annotated[str, MinLen(min_length=1), MaxLen(max_length=64)], decision_status: Annotated[str, MinLen(min_length=1), MaxLen(max_length=64)], finance_decision_id: Annotated[str, MinLen(min_length=1), MaxLen(max_length=64)], audit_event_id: Annotated[str, MinLen(min_length=1), MaxLen(max_length=64)], approval_request_id: Optional[str] = None, exception_case_id: 

## Persist the Governance Decision

The in-memory governance result must now become durable business evidence.

For a `HOLD` outcome, the persistence layer will:

1. Store the finance decision.
2. Update the invoice case status.
3. Create an approval request.
4. Record governance audit evidence.
5. Preserve the case and correlation identifiers.
6. Protect against repeated persistence.

In [21]:
import sqlite3


async def resolve_result(value):
    """Support synchronous or asynchronous service methods."""
    if inspect.isawaitable(value):
        return await value
    return value


print("=" * 90)
print("PERSIST GOVERNANCE HOLD")
print("=" * 90)
print(f"Case reference    : {case_reference}")
print(f"Correlation ID    : {correlation_id}")
print(f"Governance status : {governance_status}")
print(
    f"Required authority: "
    f"{governance_required_authority}"
)
print("-" * 90)

assert governance_status == "HOLD", (
    "This demo step expects a governance HOLD."
)

persistence_service = GovernancePersistenceService()

persistence_result = await resolve_result(
    persistence_service.persist_decision(
        orchestration_result=governance_result,
        case_id=case_reference,
        correlation_id=correlation_id,
        invoice_case_id=case_reference,
    )
)

persistence_result_dict = object_to_dictionary(
    persistence_result
)

persistence_display = make_display_safe(
    persistence_result_dict
)

print("Finance-decision persistence result:")
pprint(persistence_display, sort_dicts=False)

finance_decision_id = persistence_result_dict.get(
    "finance_decision_id"
)

assert finance_decision_id, (
    "No finance decision ID was returned."
)

approval_creation_result = await resolve_result(
    persistence_service.persist_hold_decision(
        orchestration_result=governance_result,
        case_id=case_reference,
        correlation_id=correlation_id,
        finance_decision_id=finance_decision_id,
    )
)

if approval_creation_result is not None:
    approval_creation_dict = object_to_dictionary(
        approval_creation_result
    )

    print("\nApproval-request creation result:")
    pprint(
        make_display_safe(approval_creation_dict),
        sort_dicts=False,
    )
else:
    approval_creation_dict = {}
    print(
        "\nNo new approval object was returned. "
        "Checking for an existing approval request."
    )

identifier.unknown_dialect_budget: /Users/pmayank/workspace/LdcDemo/.venv/lib/python3.12/site-packages/dataflow/migrations/sync_ddl_executor.py:583 validated an identifier against DIALECT_UNKNOWN_MAX_IDENTIFIER_LENGTH (128, SQLite's — the LOOSEST). If this path can reach PostgreSQL, an identifier of 64-128 chars passes validation here and is TRUNCATED server-side at 63, which can alias two models onto one table (#1971). Bind the target dialect and pass its budget (dialect._MAX_IDENTIFIER_LENGTH) instead.


PERSIST GOVERNANCE HOLD
Case reference    : LOCAL-B970E86BB68E
Correlation ID    : CORR-2CBC6CC15D7249CB8EE549B035F87E41
Governance status : HOLD
Required authority: L3_CONTROLLER
------------------------------------------------------------------------------------------
Finance-decision persistence result:
{'outcome_status': 'HOLD',
 'decision_status': 'HOLD',
 'finance_decision_id': 'fin-64c702dddcf2',
 'audit_event_id': 'aud-e644c9915351',
 'approval_request_id': 'arq-6f3f897edc52',
 'exception_case_id': None,
 'invoice_id': 'LOCAL-B970E86BB68E',
 'correlation_id': 'CORR-2CBC6CC15D7249CB8EE549B035F87E41',
 'case_exists': True,
 'is_replay': False,
 'explanation': 'Decision persisted: HOLD for LOCAL-B970E86BB68E'}

Approval-request creation result:
{'approval_request_id': 'arq-6f3f897edc52',
 'invoice_id': 'LOCAL-B970E86BB68E',
 'correlation_id': 'CORR-2CBC6CC15D7249CB8EE549B035F87E41',
 'requested_from': 'L3_CONTROLLER',
 'approval_reason': 'GOVERNANCE_PENDING',
 'created_at': '2026-

In [ ]:
database_uri = f"{DATABASE_PATH.resolve().as_uri()}?mode=ro"

with sqlite3.connect(
    database_uri,
    uri=True,
) as connection:
    connection.row_factory = sqlite3.Row
    cursor = connection.cursor()

    finance_decision_rows = cursor.execute(
        """
        SELECT
            id,
            invoice_id,
            correlation_id,
            decision_type,
            recommended_value,
            confidence,
            rationale,
            agent_id,
            policy_outcome,
            decision_timestamp
        FROM finance_decisions
        WHERE invoice_id = ?
          AND correlation_id = ?
        ORDER BY created_at
        """,
        (
            case_reference,
            correlation_id,
        ),
    ).fetchall()

    approval_request_rows = cursor.execute(
        """
        SELECT
            id,
            invoice_id,
            correlation_id,
            requested_by,
            requested_from,
            approval_reason,
            approval_status,
            requested_at
        FROM approval_requests
        WHERE invoice_id = ?
          AND correlation_id = ?
        ORDER BY created_at
        """,
        (
            case_reference,
            correlation_id,
        ),
    ).fetchall()

    invoice_case_row = cursor.execute(
        """
        SELECT
            id,
            correlation_id,
            target_system,
            case_status,
            current_owner,
            updated_at
        FROM invoice_cases
        WHERE id = ?
        """,
        (case_reference,),
    ).fetchone()

    governance_audit_rows = cursor.execute(
        """
        SELECT
            id,
            invoice_id,
            correlation_id,
            actor_id,
            action_type,
            action_outcome,
            event_timestamp
        FROM business_audit_events
        WHERE invoice_id = ?
          AND correlation_id = ?
        ORDER BY created_at
        """,
        (
            case_reference,
            correlation_id,
        ),
    ).fetchall()

finance_decision_evidence = [
    dict(row) for row in finance_decision_rows
]

approval_request_evidence = [
    dict(row) for row in approval_request_rows
]

invoice_case_evidence = (
    dict(invoice_case_row)
    if invoice_case_row
    else None
)

governance_audit_evidence = [
    dict(row) for row in governance_audit_rows
]

print("=" * 90)
print("PERSISTED GOVERNANCE EVIDENCE")
print("=" * 90)

print("\nFINANCE DECISION")
for row in finance_decision_evidence:
    print(f"  Decision ID    : {row['id']}")
    print(f"  Decision type  : {row['decision_type']}")
    print(f"  Recommended    : {row['recommended_value']}")
    print(f"  Confidence     : {row['confidence']}")
    print(f"  Policy outcome : {row['policy_outcome']}")
    print(f"  Rationale      : {row['rationale']}")

print("\nAPPROVAL REQUEST")
for row in approval_request_evidence:
    print(f"  Request ID     : {row['id']}")
    print(f"  Requested from : {row['requested_from']}")
    print(f"  Status         : {row['approval_status']}")
    print(f"  Reason         : {row['approval_reason']}")

print("\nINVOICE CASE")
if invoice_case_evidence:
    print(
        f"  Case status    : "
        f"{invoice_case_evidence['case_status']}"
    )
    print(
        f"  Target system  : "
        f"{invoice_case_evidence['target_system']}"
    )
    print(
        f"  Current owner  : "
        f"{invoice_case_evidence['current_owner']}"
    )

finance_decision_id = (
    finance_decision_evidence[0]["id"]
    if finance_decision_evidence
    else finance_decision_id
)

approval_request_id = (
    approval_request_evidence[0]["id"]
    if approval_request_evidence
    else approval_creation_dict.get(
        "approval_request_id"
    )
)

governance_persistence_checks = {
    "Exactly one finance decision exists": (
        len(finance_decision_evidence) == 1
    ),
    "Exactly one approval request exists": (
        len(approval_request_evidence) == 1
    ),
    "Finance decision records HOLD": (
        finance_decision_evidence
        and finance_decision_evidence[0][
            "policy_outcome"
        ]
        == "HOLD"
    ),
    "Approval request is pending": (
        approval_request_evidence
        and approval_request_evidence[0][
            "approval_status"
        ]
        == "PENDING"
    ),
    "Invoice case exists": (
        invoice_case_evidence is not None
    ),
    "Invoice case remains in a controlled state": (
    invoice_case_evidence
    and invoice_case_evidence["case_status"]
    in {
        "RECEIVED",
        "HOLD",
        "PENDING_APPROVAL",
    }
),
"Posting is blocked while approval is pending": (
    posting_count_before_approval == 0
),
    "Finance decision ID is available": (
        bool(finance_decision_id)
    ),
    "Approval request ID is available": (
        bool(approval_request_id)
    ),
    "Governance audit evidence exists": (
        len(governance_audit_evidence) >= 1
    ),
}

print("\n" + "=" * 90)
print("GOVERNANCE PERSISTENCE VALIDATION")
print("=" * 90)

for check_name, passed in governance_persistence_checks.items():
    symbol = "✅" if passed else "❌"
    status = "PASS" if passed else "FAIL"
    print(f"{symbol} {status:<4} | {check_name}")

governance_hold_persisted = all(
    governance_persistence_checks.values()
)

print("-" * 90)

if governance_hold_persisted:
    print("✅ GOVERNANCE HOLD PERSISTED")
    print("✅ APPROVAL REQUEST CREATED")
    print(f"Approval request ID: {approval_request_id}")
else:
    print("❌ GOVERNANCE PERSISTENCE VALIDATION FAILED")

assert governance_hold_persisted, (
    "The governance HOLD and approval request "
    "were not persisted as expected."
)

PERSISTED GOVERNANCE EVIDENCE

FINANCE DECISION
  Decision ID    : fin-64c702dddcf2
  Decision type  : GOVERNANCE
  Recommended    : HOLD
  Confidence     : 1.0
  Policy outcome : HOLD
  Rationale      : Aegis holding: Request REQ-LOCAL-B970E86BB68E has conditional approval

APPROVAL REQUEST
  Request ID     : arq-6f3f897edc52
  Requested from : L3_CONTROLLER
  Status         : PENDING
  Reason         : GOVERNANCE_PENDING

INVOICE CASE
  Case status    : RECEIVED
  Target system  : 
  Current owner  : finance-orchestrator

GOVERNANCE PERSISTENCE VALIDATION
✅ PASS | Exactly one finance decision exists
✅ PASS | Exactly one approval request exists
✅ PASS | Finance decision records HOLD
✅ PASS | Approval request is pending
✅ PASS | Invoice case exists
❌ FAIL | Invoice case is on HOLD
✅ PASS | Finance decision ID is available
✅ PASS | Approval request ID is available
✅ PASS | Governance audit evidence exists
----------------------------------------------------------------------------------

AssertionError: The governance HOLD and approval request were not persisted as expected.